# AutoWorth — Used Car Price Prediction

This notebook reproduces the data preparation and model comparison used by the Streamlit app. Dataset: [CarDekho public listings](https://github.com/bagassenop/cardekho/blob/main/CAR%20DETAILS%20FROM%20CAR%20DEKHO.csv).

In [ ]:
import pandas as pd
import urllib.request
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

url = 'https://raw.githubusercontent.com/bagassenop/cardekho/main/CAR%20DETAILS%20FROM%20CAR%20DEKHO.csv'
with urllib.request.urlopen(url) as response:
    df = pd.read_csv(response)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df.head()

In [ ]:
df['model_name'] = df['name'].astype(str).str.strip().str.title()
df['brand'] = df['model_name'].str.split().str[0]
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['km_driven'] = pd.to_numeric(df['km_driven'], errors='coerce')
df['selling_price'] = pd.to_numeric(df['selling_price'], errors='coerce')
owner_map = {'First Owner':0, 'Second Owner':1, 'Third Owner':2, 'Fourth & Above Owner':3}
df['owner'] = df['owner'].map(owner_map)
df = df.dropna(subset=['selling_price','year','km_driven'])
df = df[df.selling_price > 0]
features = ['brand','model_name','year','km_driven','fuel','seller_type','transmission','owner']
numeric = ['year','km_driven','owner']
categorical = ['brand','model_name','fuel','seller_type','transmission']
X, y = df[features], df['selling_price']
print('Rows:', len(df), '| Median price:', y.median())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
def build_model(estimator):
    prep = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ])
    return Pipeline([('prep', prep), ('model', estimator)])
models = {
    'Random Forest': build_model(RandomForestRegressor(n_estimators=260, min_samples_leaf=2, max_features=.9, n_jobs=-1, random_state=42)),
    'Extra Trees': build_model(ExtraTreesRegressor(n_estimators=260, min_samples_leaf=2, max_features=.9, n_jobs=-1, random_state=42)),
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({'model':name, 'MAE (₹)':mean_absolute_error(y_test,pred), 'R²':r2_score(y_test,pred)})
pd.DataFrame(results).sort_values('MAE (₹)')

## Interpretation

Use the displayed held-out MAE as a typical absolute prediction error, not a guaranteed price range. Results depend on the dataset version. Condition, location, exact variant, and service records are not represented in the input features.